# ETL — Localização PT-BR do Dataset de Sinistros de Saúde

Pipeline completo de enriquecimento e localização do dataset `enhanced_health_insurance_claims.csv` para o contexto brasileiro.

In [33]:
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

# Link do dataset original : https://www.kaggle.com/datasets/leandrenash/enhanced-health-insurance-claims-dataset

BASE = Path('../datasets')
INPUT = BASE / 'archive/enhanced_health_insurance_claims.csv'
OUTPUT_DIR = BASE / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

USD_TO_BRL = 5.75
SEED = 42
rng = np.random.default_rng(SEED)

print('Setup concluído.')
print(f'Output dir: {OUTPUT_DIR.resolve()}')

Setup concluído.
Output dir: /Users/pedrocunha/Projects/Personal/ai-operador-medico/datasets/output


## Célula 1 — Carregar Dataset

In [34]:
df = pd.read_csv(INPUT)
print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head(3)

Shape: (4500, 17)
Colunas: ['ClaimID', 'PatientID', 'ProviderID', 'ClaimAmount', 'ClaimDate', 'DiagnosisCode', 'ProcedureCode', 'PatientAge', 'PatientGender', 'ProviderSpecialty', 'ClaimStatus', 'PatientIncome', 'PatientMaritalStatus', 'PatientEmploymentStatus', 'ProviderLocation', 'ClaimType', 'ClaimSubmissionMethod']


,ClaimID,PatientID,ProviderID,ClaimAmount,ClaimDate,DiagnosisCode,ProcedureCode,PatientAge,PatientGender,ProviderSpecialty,ClaimStatus,PatientIncome,PatientMaritalStatus,PatientEmploymentStatus,ProviderLocation,ClaimType,ClaimSubmissionMethod
0,10944daf-f7d5-4e1d-8216-72ffa609fe41,8552381d-7960-4f64-b190-b20b8ada00a1,4a4cb19c-4863-41cf-84b0-c2b21aace988,3807.95,2024-06-07,yy006,hd662,16,M,Cardiology,Pending,90279.43,Married,Retired,Jameshaven,Routine,Paper
1,fcbebb25-fc24-4c0f-a966-749edcf83fb1,327f43ad-e3bd-4473-a9ed-46483a0a156f,422e02dd-c1fd-43dd-8af4-0c3523f997b1,9512.07,2023-05-30,tD052,mH831,27,M,Pediatrics,Approved,130448.02,Single,Student,Beltrantown,Routine,Online
2,9e9983e7-9ea7-45f5-84d8-ce49ccd8a4a1,6f3acdf7-73aa-4afa-9c2e-b25b27bdb5b0,f7733b3f-0980-47b5-a7a0-ee390869355b,7346.74,2022-09-27,zx832,dg637,40,F,Cardiology,Pending,82417.54,Divorced,Employed,West Charlesport,Emergency,Online


## Célula 2 — Conversão Monetária (USD → BRL)

In [35]:
df = df.rename(columns={
    'ClaimAmount': 'ClaimAmountUSD',
    'PatientIncome': 'PatientIncomeUSD'
})
df['ClaimAmountBRL'] = (df['ClaimAmountUSD'] * USD_TO_BRL).round(2)
df['PatientIncomeBRL'] = (df['PatientIncomeUSD'] * USD_TO_BRL).round(2)

print('Conversão monetária concluída.')
print(f'ClaimAmountUSD range: {df["ClaimAmountUSD"].min():.2f} - {df["ClaimAmountUSD"].max():.2f}')
print(f'ClaimAmountBRL range: {df["ClaimAmountBRL"].min():.2f} - {df["ClaimAmountBRL"].max():.2f}')
print(f'PatientIncomeBRL range: {df["PatientIncomeBRL"].min():.2f} - {df["PatientIncomeBRL"].max():.2f}')

Conversão monetária concluída.
ClaimAmountUSD range: 100.12 - 9997.20
ClaimAmountBRL range: 575.69 - 57483.90
PatientIncomeBRL range: 115039.50 - 862255.74


## Célula 3 — Traduções PT-BR

In [36]:
gender_map = {'M': 'Masculino', 'F': 'Feminino'}
marital_map = {
    'Single': 'Solteiro(a)', 'Married': 'Casado(a)',
    'Divorced': 'Divorciado(a)', 'Widowed': 'Viúvo(a)'
}
employment_map = {
    'Employed': 'Empregado', 'Unemployed': 'Desempregado',
    'Retired': 'Aposentado', 'Student': 'Estudante'
}
claim_type_map = {
    'Inpatient': 'Internação', 'Outpatient': 'Ambulatorial',
    'Emergency': 'Emergência', 'Routine': 'Consulta de Rotina'
}
submission_map = {'Online': 'Online', 'Paper': 'Papel/Correio', 'Phone': 'Telefone'}
specialty_map = {
    'Cardiology': 'Cardiologia', 'General Practice': 'Clínica Geral',
    'Neurology': 'Neurologia', 'Orthopedics': 'Ortopedia', 'Pediatrics': 'Pediatria'
}

df['PatientGender'] = df['PatientGender'].map(gender_map)
df['PatientMaritalStatus'] = df['PatientMaritalStatus'].map(marital_map)
df['PatientEmploymentStatus'] = df['PatientEmploymentStatus'].map(employment_map)
df['ClaimType'] = df['ClaimType'].map(claim_type_map)
df['ClaimSubmissionMethod'] = df['ClaimSubmissionMethod'].map(submission_map)
df['ProviderSpecialty'] = df['ProviderSpecialty'].map(specialty_map)

# ClaimStatus será calculado com regras de negócio após as colunas
# sintéticas serem geradas (célula 6.5)
df = df.drop(columns=['ClaimStatus'])

print('Traduções PT-BR aplicadas.')
for col in ['PatientGender', 'PatientMaritalStatus', 'PatientEmploymentStatus',
            'ClaimType', 'ClaimSubmissionMethod', 'ProviderSpecialty']:
    print(f'  {col}: {df[col].unique().tolist()}')

Traduções PT-BR aplicadas.
  PatientGender: ['Masculino', 'Feminino']
  PatientMaritalStatus: ['Casado(a)', 'Solteiro(a)', 'Divorciado(a)', 'Viúvo(a)']
  PatientEmploymentStatus: ['Aposentado', 'Estudante', 'Empregado', 'Desempregado']
  ClaimType: ['Consulta de Rotina', 'Emergência', 'Internação', 'Ambulatorial']
  ClaimSubmissionMethod: ['Papel/Correio', 'Online', 'Telefone']
  ProviderSpecialty: ['Cardiologia', 'Pediatria', 'Neurologia', 'Clínica Geral', 'Ortopedia']


## Célula 4 — Localização ProviderLocation → Estados Brasileiros

In [37]:
STATES = [
    ('SP', 'São Paulo', 22.0), ('MG', 'Minas Gerais', 10.0),
    ('RJ', 'Rio de Janeiro', 8.0), ('BA', 'Bahia', 7.0),
    ('PR', 'Paraná', 5.5), ('RS', 'Rio Grande do Sul', 5.5),
    ('PE', 'Pernambuco', 4.5), ('CE', 'Ceará', 4.5),
    ('PA', 'Pará', 4.0), ('MA', 'Maranhão', 3.5),
    ('GO', 'Goiás', 3.0), ('SC', 'Santa Catarina', 3.0),
    ('AM', 'Amazonas', 2.0), ('PB', 'Paraíba', 2.0),
    ('ES', 'Espírito Santo', 2.0), ('RN', 'Rio Grande do Norte', 1.7),
    ('MT', 'Mato Grosso', 1.7), ('AL', 'Alagoas', 1.5),
    ('MS', 'Mato Grosso do Sul', 1.5), ('DF', 'Distrito Federal', 1.5),
    ('PI', 'Piauí', 1.2), ('RO', 'Rondônia', 0.8),
    ('TO', 'Tocantins', 0.8), ('SE', 'Sergipe', 0.6),
    ('AC', 'Acre', 0.4), ('AP', 'Amapá', 0.4), ('RR', 'Roraima', 0.4),
]

total_weight = sum(w for _, _, w in STATES)
cumulative = []
acc = 0.0
for sigla, nome, peso in STATES:
    acc += (peso / total_weight) * 10000
    cumulative.append((sigla, nome, acc))

def assign_state(location: str):
    h = int(hashlib.md5(location.encode()).hexdigest(), 16) % 10000
    for sigla, nome, threshold in cumulative:
        if h < threshold:
            return sigla, nome
    return cumulative[-1][0], cumulative[-1][1]

location_cache = {}
def get_state(location):
    if location not in location_cache:
        location_cache[location] = assign_state(location)
    return location_cache[location]

df['ProviderState'] = df['ProviderLocation'].apply(lambda x: get_state(x)[0])
df['ProviderStateName'] = df['ProviderLocation'].apply(lambda x: get_state(x)[1])

print('Localização por estados brasileiros concluída.')
print('Distribuição por estado (top 10):')
print(df['ProviderState'].value_counts().head(10))

Localização por estados brasileiros concluída.
Distribuição por estado (top 10):
ProviderState
SP    975
MG    450
RJ    366
BA    312
PR    257
PE    228
RS    214
PA    202
CE    186
MA    152
Name: count, dtype: int64


## Célula 5 — Tabelas Auxiliares: diagnosis_codes.csv e procedure_codes.csv

In [38]:
DIAGNOSIS_POOLS = {
    'Cardiologia': [
        ('Insuficiência Cardíaca Congestiva', 'Doenças Cardiovasculares'),
        ('Hipertensão Arterial Sistêmica', 'Doenças Cardiovasculares'),
        ('Infarto Agudo do Miocárdio', 'Doenças Cardiovasculares'),
        ('Arritmia Cardíaca', 'Doenças Cardiovasculares'),
        ('Angina Pectoris Instável', 'Doenças Cardiovasculares'),
        ('Fibrilação Atrial', 'Doenças Cardiovasculares'),
        ('Cardiomiopatia Dilatada', 'Doenças Cardiovasculares'),
        ('Valvopatia Mitral', 'Doenças Cardiovasculares'),
        ('Endocardite Infecciosa', 'Doenças Cardiovasculares'),
        ('Trombose Venosa Profunda', 'Doenças Vasculares'),
        ('Embolia Pulmonar', 'Doenças Vasculares'),
        ('Aterosclerose Coronariana', 'Doenças Cardiovasculares'),
        ('Pericardite Aguda', 'Doenças Cardiovasculares'),
        ('Choque Cardiogênico', 'Doenças Cardiovasculares'),
    ],
    'Clínica Geral': [
        ('Diabetes Mellitus Tipo 2', 'Doenças Metabólicas'),
        ('Hipertensão Arterial Sistêmica', 'Doenças Cardiovasculares'),
        ('Infecção do Trato Urinário', 'Doenças Infecciosas'),
        ('Pneumonia Bacteriana', 'Doenças Respiratórias'),
        ('Gastroenterite Aguda', 'Doenças Gastrointestinais'),
        ('Rinite Alérgica', 'Doenças Alérgicas'),
        ('Lombalgia Crônica', 'Doenças Musculoesqueléticas'),
        ('Hipotireoidismo', 'Doenças Endócrinas'),
        ('Anemia Ferropriva', 'Doenças Hematológicas'),
        ('Depressão Maior', 'Transtornos Mentais'),
        ('Ansiedade Generalizada', 'Transtornos Mentais'),
        ('Síndrome do Intestino Irritável', 'Doenças Gastrointestinais'),
        ('Obesidade Grau II', 'Doenças Metabólicas'),
        ('Doença do Refluxo Gastroesofágico', 'Doenças Gastrointestinais'),
        ('Bronquite Crônica', 'Doenças Respiratórias'),
    ],
    'Neurologia': [
        ('Enxaqueca Crônica', 'Doenças Neurológicas'),
        ('Epilepsia', 'Doenças Neurológicas'),
        ('Acidente Vascular Cerebral Isquêmico', 'Doenças Cerebrovasculares'),
        ('Doença de Parkinson', 'Doenças Neurodegenerativas'),
        ('Esclerose Múltipla', 'Doenças Autoimunes Neurológicas'),
        ('Neuropatia Periférica Diabética', 'Doenças Neurológicas'),
        ('Demência de Alzheimer', 'Doenças Neurodegenerativas'),
        ('Cefaleia Tensional Crônica', 'Doenças Neurológicas'),
        ('Neuralgia do Trigêmeo', 'Doenças Neurológicas'),
        ('Síndrome de Guillain-Barré', 'Doenças Neurológicas'),
        ('Meningite Bacteriana', 'Doenças Infecciosas do SNC'),
        ('Acidente Isquêmico Transitório', 'Doenças Cerebrovasculares'),
        ('Tremor Essencial', 'Doenças Neurológicas'),
        ('Miastenia Gravis', 'Doenças Neuromusculares'),
    ],
    'Ortopedia': [
        ('Fratura de Fêmur', 'Traumatologia'),
        ('Lesão do Manguito Rotador', 'Doenças Musculoesqueléticas'),
        ('Artrose de Joelho', 'Doenças Articulares'),
        ('Hérnia de Disco Lombar', 'Doenças da Coluna'),
        ('Tendinite Patelar', 'Doenças Musculoesqueléticas'),
        ('Escoliose Idiopática', 'Doenças da Coluna'),
        ('Fratura de Punho', 'Traumatologia'),
        ('Osteoporose', 'Doenças Ósseas'),
        ('Artrite Reumatoide', 'Doenças Autoimunes'),
        ('Síndrome do Túnel do Carpo', 'Doenças Nervosas Periféricas'),
        ('Bursite do Quadril', 'Doenças Musculoesqueléticas'),
        ('Fratura de Tornozelo', 'Traumatologia'),
        ('Tendinite de Aquiles', 'Doenças Musculoesqueléticas'),
        ('Epicondilite Lateral', 'Doenças Musculoesqueléticas'),
    ],
    'Pediatria': [
        ('Bronquiolite Aguda', 'Doenças Respiratórias Pediátricas'),
        ('Asma Infantil', 'Doenças Respiratórias Pediátricas'),
        ('Otite Média Aguda', 'Doenças Otorrinolaringológicas'),
        ('Amigdalite Bacteriana', 'Doenças Otorrinolaringológicas'),
        ('Dermatite Atópica', 'Doenças Dermatológicas'),
        ('Varicela', 'Doenças Infecciosas Pediátricas'),
        ('Pneumonia Viral', 'Doenças Respiratórias Pediátricas'),
        ('Gastroenterite Aguda Pediátrica', 'Doenças Gastrointestinais'),
        ('Febre Reumática', 'Doenças Infecciosas Pediátricas'),
        ('Convulsão Febril', 'Doenças Neurológicas Pediátricas'),
        ('Síndrome do Croup', 'Doenças Respiratórias Pediátricas'),
        ('Rinite Alérgica Pediátrica', 'Doenças Alérgicas'),
        ('Anemia Falciforme', 'Doenças Hematológicas'),
        ('Diabetes Mellitus Tipo 1', 'Doenças Metabólicas'),
    ],
}

PROCEDURE_POOLS = {
    'Cardiologia': [
        ('Cateterismo Cardíaco', 'Procedimentos Hemodinâmicos'),
        ('Ecocardiograma Transtorácico', 'Diagnóstico por Imagem'),
        ('Holter 24 Horas', 'Monitorização Cardíaca'),
        ('Cardioversão Elétrica', 'Procedimentos Terapêuticos'),
        ('Implante de Marcapasso', 'Cirurgia Cardíaca'),
        ('Angioplastia Coronariana', 'Procedimentos Hemodinâmicos'),
        ('Revascularização do Miocárdio', 'Cirurgia Cardíaca'),
        ('Ecocardiograma de Estresse', 'Diagnóstico por Imagem'),
        ('Monitorização de Pressão Arterial (MAPA)', 'Monitorização Cardíaca'),
        ('Ablação por Radiofrequência', 'Procedimentos Eletrofisiológicos'),
        ('Teste Ergométrico', 'Diagnóstico Funcional'),
        ('Implante de Desfibrilador', 'Cirurgia Cardíaca'),
        ('Ecodopplercardiograma', 'Diagnóstico por Imagem'),
        ('Valvoplastia Mitral', 'Cirurgia Cardíaca'),
    ],
    'Clínica Geral': [
        ('Consulta Médica Ambulatorial', 'Consultas'),
        ('Coleta de Exames Laboratoriais', 'Diagnóstico Laboratorial'),
        ('Eletrocardiograma', 'Diagnóstico Cardiológico'),
        ('Espirometria', 'Diagnóstico Respiratório'),
        ('Ultrassonografia Abdominal', 'Diagnóstico por Imagem'),
        ('Raio-X de Tórax', 'Diagnóstico por Imagem'),
        ('Curativo e Sutura', 'Procedimentos Ambulatoriais'),
        ('Hemograma Completo', 'Diagnóstico Laboratorial'),
        ('Medição de Glicemia Capilar', 'Monitorização Metabólica'),
        ('Vacinação', 'Medicina Preventiva'),
        ('Aferição de Pressão Arterial', 'Monitorização Cardiovascular'),
        ('Teste de Gravidez', 'Diagnóstico Laboratorial'),
        ('Nebulização', 'Procedimentos Terapêuticos'),
        ('Orientação Nutricional', 'Promoção da Saúde'),
        ('Eletroencefalograma', 'Diagnóstico Neurológico'),
    ],
    'Neurologia': [
        ('Eletroencefalograma (EEG)', 'Diagnóstico Neurológico'),
        ('Ressonância Magnética Encefálica', 'Diagnóstico por Imagem'),
        ('Punção Lombar', 'Procedimentos Diagnósticos'),
        ('Eletroneuromiografia', 'Diagnóstico Neuromuscular'),
        ('Potencial Evocado Visual', 'Diagnóstico Neurofisiológico'),
        ('Tomografia de Crânio', 'Diagnóstico por Imagem'),
        ('Doppler Transcraniano', 'Diagnóstico Vascular'),
        ('Avaliação Neuropsicológica', 'Avaliação Cognitiva'),
        ('Toxina Botulínica Terapêutica', 'Procedimentos Terapêuticos'),
        ('Estimulação Magnética Transcraniana', 'Procedimentos Terapêuticos'),
        ('Biópsia de Nervo Periférico', 'Procedimentos Cirúrgicos'),
        ('Angiografia Cerebral', 'Diagnóstico por Imagem'),
        ('Polissonografia', 'Diagnóstico do Sono'),
        ('Implante de Estimulador Cerebral Profundo', 'Neurocirurgia'),
    ],
    'Ortopedia': [
        ('Artroscopia de Joelho', 'Cirurgia Artroscópica'),
        ('Artroplastia Total de Quadril', 'Cirurgia Articular'),
        ('Osteossíntese de Fêmur', 'Procedimentos Cirúrgicos'),
        ('Artroscopia de Ombro', 'Cirurgia Artroscópica'),
        ('Artroplastia Total de Joelho', 'Cirurgia Articular'),
        ('Discectomia Lombar', 'Cirurgia da Coluna'),
        ('Fisioterapia Ortopédica', 'Reabilitação'),
        ('Imobilização com Gesso', 'Procedimentos Ambulatoriais'),
        ('Infiltração Intra-articular', 'Procedimentos Terapêuticos'),
        ('Osteossíntese de Punho', 'Procedimentos Cirúrgicos'),
        ('Tenotomia', 'Procedimentos Cirúrgicos'),
        ('Radiografia Óssea', 'Diagnóstico por Imagem'),
        ('Densitometria Óssea', 'Diagnóstico por Imagem'),
        ('Correção Cirúrgica de Escoliose', 'Cirurgia da Coluna'),
    ],
    'Pediatria': [
        ('Teste do Pezinho', 'Triagem Neonatal'),
        ('Vacinação Pediátrica', 'Medicina Preventiva'),
        ('Nebulização Pediátrica', 'Procedimentos Terapêuticos'),
        ('Triagem Auditiva Neonatal', 'Triagem Neonatal'),
        ('Consulta de Puericultura', 'Consultas'),
        ('Teste do Olhinho', 'Triagem Neonatal'),
        ('Amigdalectomia', 'Cirurgia Pediátrica'),
        ('Adenoidectomia', 'Cirurgia Pediátrica'),
        ('Oximetria de Pulso', 'Monitorização Respiratória'),
        ('Ventilação Não Invasiva Pediátrica', 'Suporte Ventilatório'),
        ('Paracentese de Membrana Timpânica', 'Cirurgia Otorrinolaringológica'),
        ('Avaliação do Desenvolvimento Neuropsicomotor', 'Avaliação Pediátrica'),
        ('Coleta de Hemocultura Pediátrica', 'Diagnóstico Laboratorial'),
        ('Fototerapia para Icterícia Neonatal', 'Procedimentos Terapêuticos'),
    ],
}

def get_diagnosis(code: str, specialty: str):
    pool = DIAGNOSIS_POOLS.get(specialty, DIAGNOSIS_POOLS['Clínica Geral'])
    idx = int(hashlib.md5(code.encode()).hexdigest(), 16) % len(pool)
    return pool[idx]

def get_procedure(code: str, specialty: str):
    pool = PROCEDURE_POOLS.get(specialty, PROCEDURE_POOLS['Clínica Geral'])
    idx = int(hashlib.md5(code.encode()).hexdigest(), 16) % len(pool)
    return pool[idx]

# Tabela diagnosis_codes
diag_source = df[['DiagnosisCode', 'ProviderSpecialty']].drop_duplicates('DiagnosisCode')
diag_rows = []
for _, row in diag_source.iterrows():
    name, cat = get_diagnosis(row['DiagnosisCode'], row['ProviderSpecialty'])
    diag_rows.append({'DiagnosisCode': row['DiagnosisCode'], 'DiagnosisNamePTBR': name, 'DiagnosisCategory': cat})
diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(OUTPUT_DIR / 'diagnosis_codes.csv', index=False, encoding='utf-8-sig')
print(f'diagnosis_codes.csv: {len(diag_df)} registros')

# Tabela procedure_codes
proc_source = df[['ProcedureCode', 'ProviderSpecialty']].drop_duplicates('ProcedureCode')
proc_rows = []
for _, row in proc_source.iterrows():
    name, cat = get_procedure(row['ProcedureCode'], row['ProviderSpecialty'])
    proc_rows.append({'ProcedureCode': row['ProcedureCode'], 'ProcedureNamePTBR': name, 'ProcedureCategory': cat})
proc_df = pd.DataFrame(proc_rows)
proc_df.to_csv(OUTPUT_DIR / 'procedure_codes.csv', index=False, encoding='utf-8-sig')
print(f'procedure_codes.csv: {len(proc_df)} registros')

# Join com dataset principal
df = df.merge(diag_df[['DiagnosisCode', 'DiagnosisNamePTBR']], on='DiagnosisCode', how='left')
df = df.merge(proc_df[['ProcedureCode', 'ProcedureNamePTBR']], on='ProcedureCode', how='left')
print(f'Join concluído. Shape após join: {df.shape}')

diagnosis_codes.csv: 4495 registros
procedure_codes.csv: 4495 registros
Join concluído. Shape após join: (4500, 22)


## Célula 6 — Novas Colunas de Perfil do Paciente

In [39]:
# ChildrenCount
def get_children_lambda(row):
    status = row['PatientMaritalStatus']
    age = row['PatientAge']
    if status == 'Solteiro(a)':
        return 0.2 if age < 30 else 0.6
    elif status == 'Casado(a)':
        if age < 30: return 0.8
        elif age <= 45: return 1.8
        else: return 2.2
    else:  # Divorciado(a) ou Viúvo(a)
        return 1.2

lambdas = df.apply(get_children_lambda, axis=1).values
df['ChildrenCount'] = rng.poisson(lambdas).clip(0, 5).astype(int)

# IsHomeOwner
def get_home_prob(row):
    income = row['PatientIncomeUSD']
    emp = row['PatientEmploymentStatus']
    prob = 0.30 if income < 50000 else (0.55 if income <= 100000 else 0.75)
    if emp == 'Aposentado': prob += 0.15
    elif emp in ('Desempregado', 'Estudante'): prob -= 0.15
    return max(0.05, min(0.95, prob))

home_probs = df.apply(get_home_prob, axis=1).values
df['IsHomeOwner'] = rng.random(len(df)) < home_probs

# EducationLevel
edu_levels = ['Fundamental', 'Médio', 'Superior', 'Pós-graduação']

def get_edu_probs(row):
    emp = row['PatientEmploymentStatus']
    income = row['PatientIncomeUSD']
    if emp == 'Estudante': return [0.0, 0.80, 0.20, 0.0]
    elif emp == 'Desempregado': return [0.40, 0.40, 0.18, 0.02]
    elif emp == 'Aposentado':
        if income < 50000: return [0.25, 0.45, 0.25, 0.05]
        elif income <= 100000: return [0.08, 0.30, 0.45, 0.17]
        else: return [0.02, 0.15, 0.43, 0.40]
    else:  # Empregado
        if income < 50000: return [0.20, 0.45, 0.30, 0.05]
        elif income <= 100000: return [0.05, 0.25, 0.50, 0.20]
        else: return [0.01, 0.10, 0.45, 0.44]

edu_choices = [rng.choice(edu_levels, p=get_edu_probs(row)) for _, row in df.iterrows()]
df['EducationLevel'] = edu_choices

# PlanType
plan_types = ['Individual', 'Familiar', 'Empresarial', 'MEI']
plan_probs_map = {
    'Empregado':    [0.15, 0.20, 0.55, 0.10],
    'Desempregado': [0.40, 0.35, 0.15, 0.10],
    'Aposentado':   [0.45, 0.40, 0.10, 0.05],
    'Estudante':    [0.30, 0.60, 0.08, 0.02],
}
df['PlanType'] = [
    rng.choice(plan_types, p=plan_probs_map.get(emp, [0.25, 0.25, 0.25, 0.25]))
    for emp in df['PatientEmploymentStatus']
]

# YearsAsInsured
def get_insured_range(age):
    low = max(1, age - 18)
    high = min(30, age - 5)
    return (low, low) if high < low else (low, high)

df['YearsAsInsured'] = [
    int(rng.integers(low, high + 1))
    for low, high in (get_insured_range(age) for age in df['PatientAge'])
]

# HasChronicCondition
def get_chronic_prob(row):
    age = row['PatientAge']
    specialty = row['ProviderSpecialty']
    prob = 0.10 if age < 30 else (0.25 if age <= 50 else (0.50 if age <= 65 else 0.75))
    if specialty in ('Cardiologia', 'Neurologia'): prob += 0.20
    elif specialty == 'Clínica Geral': prob += 0.10
    return max(0.05, min(0.95, prob))

chronic_probs = df.apply(get_chronic_prob, axis=1).values
df['HasChronicCondition'] = rng.random(len(df)) < chronic_probs

print('Novas colunas de perfil adicionadas.')
print(df[['ChildrenCount', 'IsHomeOwner', 'EducationLevel', 'PlanType', 'YearsAsInsured', 'HasChronicCondition']].describe(include='all'))

Novas colunas de perfil adicionadas.
        ChildrenCount IsHomeOwner EducationLevel  PlanType  YearsAsInsured  \
count     4500.000000        4500           4500      4500     4500.000000   
unique            NaN           2              4         4             NaN   
top               NaN        True          Médio  Familiar             NaN   
freq              NaN        2402           1898      1710             NaN   
mean         1.131556         NaN            NaN       NaN       35.514889   
std          1.162322         NaN            NaN       NaN       24.718491   
min          0.000000         NaN            NaN       NaN        1.000000   
25%          0.000000         NaN            NaN       NaN       13.000000   
50%          1.000000         NaN            NaN       NaN       32.500000   
75%          2.000000         NaN            NaN       NaN       57.000000   
max          5.000000         NaN            NaN       NaN       81.000000   

       HasChronicCondition

## Célula 6.5 — Engenharia do Target: ClaimStatus por Regras de Negócio

O `ClaimStatus` original do dataset é aleatório (sem correlação com as features). Aqui geramos um target realista baseado em regras de negócio do setor de planos de saúde, tornando o problema de classificação aprendível pelos modelos.

In [40]:
def compute_claim_score(row):
    score = 0.0

    # Tipo de sinistro — emergências quase sempre aprovadas
    score += {'Emergência': +0.50, 'Internação': +0.25,
              'Consulta de Rotina': 0.0, 'Ambulatorial': -0.15}.get(row['ClaimType'], 0)

    # Razão valor / renda — sinistros caros em relação à renda geram suspeita de fraude
    ratio = row['ClaimAmountUSD'] / max(row['PatientIncomeUSD'], 1000)
    if ratio > 0.20:    score -= 0.40
    elif ratio > 0.10:  score -= 0.20
    elif ratio < 0.02:  score += 0.20

    # Tempo de convênio — fidelidade reduz risco percebido
    if row['YearsAsInsured'] >= 15:   score += 0.35
    elif row['YearsAsInsured'] >= 8:  score += 0.20
    elif row['YearsAsInsured'] >= 3:  score += 0.05
    else:                              score -= 0.25

    # Condição crônica — maior custo esperado, mais escrutínio
    if row['HasChronicCondition']:    score -= 0.25

    # Tipo de plano — corporativos têm cobertura superior
    score += {'Empresarial': +0.25, 'Familiar': +0.10,
              'Individual': -0.05, 'MEI': -0.20}.get(row['PlanType'], 0)

    # Idade
    if row['PatientAge'] > 70:    score -= 0.20
    elif row['PatientAge'] > 60:  score -= 0.10
    elif row['PatientAge'] < 25:  score += 0.10

    # Método de submissão — digital tem menos erros e fraudes
    score += {'Online': +0.15, 'Telefone': 0.0,
              'Papel/Correio': -0.15}.get(row['ClaimSubmissionMethod'], 0)

    # Especialidade — procedimentos eletivos de alto custo recebem mais auditoria
    score += {'Cardiologia': -0.15, 'Ortopedia': -0.20, 'Neurologia': -0.05,
              'Clínica Geral': +0.10, 'Pediatria': +0.15}.get(row['ProviderSpecialty'], 0)

    return score

scores = df.apply(compute_claim_score, axis=1)

# Probabilidade de aprovação via função logística + ruído realista
prob_approval = 1 / (1 + np.exp(-scores * 2.5))
noise = rng.uniform(-0.10, 0.10, size=len(df))
final_prob = (prob_approval + noise).clip(0, 1)

df['ClaimStatus'] = np.where(final_prob >= 0.5, 'Aprovado', 'Negado')

print('ClaimStatus gerado por regras de negócio.')
print(df['ClaimStatus'].value_counts())
print(f'Taxa de aprovação: {(df["ClaimStatus"] == "Aprovado").mean():.1%}')

ClaimStatus gerado por regras de negócio.
ClaimStatus
Aprovado    3114
Negado      1386
Name: count, dtype: int64
Taxa de aprovação: 69.2%


## Célula 7 — Reordenar Colunas e Exportar Dataset Principal

In [41]:
FINAL_COLUMNS = [
    'ClaimID', 'PatientID', 'ProviderID',
    'ClaimAmountUSD', 'ClaimAmountBRL', 'ClaimDate',
    'DiagnosisCode', 'DiagnosisNamePTBR', 'ProcedureCode', 'ProcedureNamePTBR',
    'PatientAge', 'PatientGender', 'ProviderSpecialty',
    'ClaimStatus', 'ClaimType', 'ClaimSubmissionMethod',
    'PatientIncomeUSD', 'PatientIncomeBRL', 'PatientMaritalStatus', 'PatientEmploymentStatus',
    'ProviderLocation', 'ProviderState', 'ProviderStateName',
    'ChildrenCount', 'IsHomeOwner', 'EducationLevel', 'PlanType', 'YearsAsInsured', 'HasChronicCondition'
]

df_final = df[FINAL_COLUMNS].copy()
df_final.to_csv(OUTPUT_DIR / 'enhanced_claims_ptbr.csv', index=False, encoding='utf-8-sig')

print(f'enhanced_claims_ptbr.csv exportado.')
print(f'Shape: {df_final.shape} (esperado: 4500 linhas × {len(FINAL_COLUMNS)} colunas)')

null_counts = df_final.isna().sum()
problematic = null_counts[null_counts > 0]
if len(problematic) > 0:
    print('\n⚠️ Colunas com nulls:')
    print(problematic)
else:
    print('\n✓ Nenhum valor nulo encontrado.')

df_final.head(3)

enhanced_claims_ptbr.csv exportado.
Shape: (4500, 29) (esperado: 4500 linhas × 29 colunas)

✓ Nenhum valor nulo encontrado.


,ClaimID,PatientID,ProviderID,ClaimAmountUSD,ClaimAmountBRL,ClaimDate,DiagnosisCode,DiagnosisNamePTBR,ProcedureCode,ProcedureNamePTBR,...,PatientEmploymentStatus,ProviderLocation,ProviderState,ProviderStateName,ChildrenCount,IsHomeOwner,EducationLevel,PlanType,YearsAsInsured,HasChronicCondition
0,10944daf-f7d5-4e1d-8216-72ffa609fe41,8552381d-7960-4f64-b190-b20b8ada00a1,4a4cb19c-4863-41cf-84b0-c2b21aace988,3807.95,21895.71,2024-06-07,yy006,Embolia Pulmonar,hd662,Teste Ergométrico,...,Aposentado,Jameshaven,RJ,Rio de Janeiro,1,True,Fundamental,Individual,6,False
1,fcbebb25-fc24-4c0f-a966-749edcf83fb1,327f43ad-e3bd-4473-a9ed-46483a0a156f,422e02dd-c1fd-43dd-8af4-0c3523f997b1,9512.07,54694.40,2023-05-30,tD052,Rinite Alérgica Pediátrica,mH831,Oximetria de Pulso,...,Estudante,Beltrantown,RJ,Rio de Janeiro,1,True,Superior,Individual,22,False
2,9e9983e7-9ea7-45f5-84d8-ce49ccd8a4a1,6f3acdf7-73aa-4afa-9c2e-b25b27bdb5b0,f7733b3f-0980-47b5-a7a0-ee390869355b,7346.74,42243.76,2022-09-27,zx832,Fibrilação Atrial,dg637,Angioplastia Coronariana,...,Empregado,West Charlesport,SP,São Paulo,0,True,Fundamental,Empresarial,26,True


## Célula 8 — Gerar Relatório ETL (etl_report.md)

In [42]:
lines = []
lines.append('# Relatório ETL — Dataset de Sinistros de Saúde PT-BR\n\n')
lines.append(f'**Data de geração:** {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}  \n')
lines.append(f'**Taxa de câmbio utilizada:** 1 USD = {USD_TO_BRL} BRL  \n')
lines.append(f'**Seed de geração (colunas sintéticas):** {SEED}  \n')
lines.append('\n---\n\n')

lines.append('## Registros Processados\n\n')
lines.append('| Métrica | Valor |\n|---|---|\n')
lines.append('| Registros no dataset original | 4.500 |\n')
lines.append(f'| Registros no dataset final | {len(df_final)} |\n')
lines.append('| Colunas originais | 17 |\n')
lines.append(f'| Colunas finais | {len(FINAL_COLUMNS)} |\n')
lines.append('| Linhas removidas | 0 |\n\n')

lines.append('## Distribuição de ClaimStatus\n\n')
lines.append('| Status | Quantidade | Percentual |\n|---|---|---|\n')
for status, count in df_final['ClaimStatus'].value_counts().items():
    pct = count / len(df_final) * 100
    lines.append(f'| {status} | {count} | {pct:.1f}% |\n')
lines.append('\n')

lines.append('## Colunas Adicionadas\n\n')
new_cols_info = [
    ('ClaimAmountBRL', 'float64', 'Valor do sinistro convertido para BRL'),
    ('PatientIncomeBRL', 'float64', 'Renda anual do paciente em BRL'),
    ('ProviderState', 'str', 'Sigla do estado brasileiro do prestador'),
    ('ProviderStateName', 'str', 'Nome completo do estado brasileiro'),
    ('DiagnosisNamePTBR', 'str', 'Nome do diagnóstico em PT-BR (baseado na especialidade)'),
    ('ProcedureNamePTBR', 'str', 'Nome do procedimento em PT-BR (baseado na especialidade)'),
    ('ChildrenCount', 'int', 'Número de filhos (0–5, correlacionado com estado civil e idade)'),
    ('IsHomeOwner', 'bool', 'Proprietário do imóvel (correlacionado com renda e emprego)'),
    ('EducationLevel', 'str', 'Nível de escolaridade (correlacionado com emprego e renda)'),
    ('PlanType', 'str', 'Tipo de plano de saúde (correlacionado com vínculo empregatício)'),
    ('YearsAsInsured', 'int', 'Anos como conveniado (1–30, correlacionado com idade)'),
    ('HasChronicCondition', 'bool', 'Possui condição crônica (correlacionado com idade e especialidade)'),
]
lines.append('| Coluna | Tipo | Descrição |\n|---|---|---|\n')
for col, dtype, desc in new_cols_info:
    lines.append(f'| {col} | {dtype} | {desc} |\n')
lines.append('\n')

lines.append('## Valores Únicos por Coluna Categórica\n\n')
cat_cols = [
    'PatientGender', 'PatientMaritalStatus', 'PatientEmploymentStatus',
    'ClaimType', 'ClaimSubmissionMethod', 'ClaimStatus', 'ProviderSpecialty',
    'ProviderState', 'EducationLevel', 'PlanType'
]
lines.append('| Coluna | Nº Únicos | Valores |\n|---|---|---|\n')
for col in cat_cols:
    unique_vals = sorted(df_final[col].dropna().unique().tolist())
    lines.append(f'| {col} | {len(unique_vals)} | {", ".join(str(v) for v in unique_vals)} |\n')
lines.append('\n')

lines.append('## Tabelas Auxiliares Geradas\n\n')
lines.append('| Arquivo | Registros | Colunas |\n|---|---|---|\n')
lines.append(f'| diagnosis_codes.csv | {len(diag_df)} | DiagnosisCode, DiagnosisNamePTBR, DiagnosisCategory |\n')
lines.append(f'| procedure_codes.csv | {len(proc_df)} | ProcedureCode, ProcedureNamePTBR, ProcedureCategory |\n')
lines.append('\n')

lines.append('## Distribuição por Especialidade × ClaimStatus\n\n')
cross = df_final.groupby(['ProviderSpecialty', 'ClaimStatus']).size().unstack(fill_value=0)
status_cols = [c for c in ['Aprovado', 'Negado'] if c in cross.columns]
cross = cross[status_cols]
lines.append('| Especialidade | ' + ' | '.join(status_cols) + ' | Total |\n')
lines.append('|---|' + '---|' * len(status_cols) + '---|\n')
for specialty in cross.index:
    row_vals = [str(cross.loc[specialty, s]) for s in status_cols]
    total = int(cross.loc[specialty].sum())
    lines.append(f'| {specialty} | ' + ' | '.join(row_vals) + f' | {total} |\n')
lines.append('\n')

lines.append('## Inconsistências Detectadas\n\n')
null_check = df_final.isna().sum()
has_nulls = null_check[null_check > 0]
if len(has_nulls) > 0:
    for col, count in has_nulls.items():
        lines.append(f'- ⚠️ **{col}**: {count} valores nulos\n')
else:
    lines.append('- ✓ Nenhuma inconsistência detectada. Todas as 4.500 linhas estão completas.\n')

report_content = ''.join(lines)
with open(OUTPUT_DIR / 'etl_report.md', 'w', encoding='utf-8') as f:
    f.write(report_content)

print('etl_report.md gerado com sucesso.')
print(f'\nArquivos em {OUTPUT_DIR.resolve()}:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1024:.1f} KB)')

etl_report.md gerado com sucesso.

Arquivos em /Users/pedrocunha/Projects/Personal/ai-operador-medico/datasets/output:
  diagnosis_codes.csv (236.4 KB)
  enhanced_claims_ptbr.csv (1576.1 KB)
  etl_report.md (3.0 KB)
  procedure_codes.csv (241.0 KB)
